In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import time as time

In [ ]:
#Data import
data = pd.read_csv('./breast+cancer+wisconsin+original/breast-cancer-wisconsin.data', header=None)

#Name the columns we will be working with
data.columns = ["ID", "Clump", "CellSize", "CellShape", "Adhesion",
                "EpithelialCellSize", "BareNuclei", "Chromatin",
                "Nucleoli", "Mitoses", "Class"]
print("Data shape of starting data: ", data.shape)

#Dropping the "ID" column - it's not a useful feature
data = data.drop("ID", axis=1)
print(data.Class.value_counts()) #We see how many records are in each class before removing missing values
# Replacing "?" symbols with NaN values so Pandas can drop these values.
data = data.replace("?", np.nan)
missing_per_column = data.isna().sum()
print("Missing values per column:")
print(missing_per_column)
print("Total missing:", missing_per_column.sum())
data = data.dropna()
print(data.Class.value_counts()) #We see how many records are left in each class after missing value removal
#Force transform all values to numeric values, preventing string values 
data = data.apply(pd.to_numeric)

#Mapping the Class feature -> transforming 2nd and 4th class to 0 and 1, this now becomes binary classification problem
data["Class"] = data["Class"].map({2: 0, 4: 1})

#Shufling the data, so one class records don't proceed the other class
shuffledData = data.sample(frac=1, random_state=123)

#Drop internal dataframes index, making it to default, this way if we need to access data[0], the index actually points to the first record, not the record with index 0 in the original data frame
data = shuffledData.reset_index(drop=True)

print("Data shape of cleaned data: ", data.shape)


In [ ]:
#Data splitting 80/10/10

fullLength = len(data) #Full length of available data observations
trainingDataEndIndex = int(len(data) * 0.8) #Index where training data ends, this is 80% of the data, so we have 20% left for validation and testing
validationDataEndIndex = int(len(data)* 0.9) #Index where validation data ends, this is 90% of the data, so we have 10% left for testing

features = data.drop("Class", axis=1).values #Dropping the "Class" column, as this is the label we want to predict, and transforming the remaining data frame to a numpy array
labels = data["Class"].values.reshape(-1,1) #Extracting the "Class" column as labels, and reshaping it to be a column vector instead of a row vector

features_train, labels_train = features[:trainingDataEndIndex], labels[:trainingDataEndIndex] #Extracting the training data (features/labels), which is the first 80% of the data
features_val, labels_val = features[trainingDataEndIndex:validationDataEndIndex], labels[trainingDataEndIndex:validationDataEndIndex] #Extracting the validation data (features/labels), which is the next 10% of the data, starting from the end of the training data to the end of the validation data
features_test, labels_test = features[validationDataEndIndex:], labels[validationDataEndIndex:] #Extracting the test data (features/labels), which is the last 10% of the data, starting from the end of the validation data to the end of the full data

features_train_before = features_train.copy()
features_val_before = features_val.copy()
features_test_before = features_test.copy()



feature_names = [
    "Clump", "CellSize", "CellShape", "Adhesion",
    "EpithelialCellSize", "BareNuclei", "Chromatin",
    "Nucleoli", "Mitoses"
]

def make_stats_table(X_before, X_after, feature_names, dataset_name):
    return pd.DataFrame({
        "Dataset": dataset_name,
        "Feature": feature_names,
        "Mean Before": X_before.mean(axis=0),
        "Std Before": X_before.std(axis=0),
        "Mean After": X_after.mean(axis=0),
        "Std After": X_after.std(axis=0)
    })

#Standardizing the data - we calculate the mean and standard deviation of the training data, and then we use these values to standardize the training, validation and test data. This way we ensure that all features have a mean of 0 and a standard deviation of 1, which can help with the convergence of the gradient descent algorithm and can also prevent features with larger scales from dominating the learning process. We calculate the mean and standard deviation only on the training dataset to prevent data leakage into the model validation and test sets
mean = features_train.mean(axis=0) #axis=0 means we compute per column, not per record
std = features_train.std(axis=0) #axis=0 means we compute per column, not per record

features_train = (features_train - mean) / std
features_val = (features_val - mean) / std
features_test = (features_test - mean) / std

train_stats = make_stats_table(features_train_before, features_train, feature_names, "Train")
val_stats = make_stats_table(features_val_before, features_val, feature_names, "Validation")
test_stats = make_stats_table(features_test_before, features_test, feature_names, "Test")

all_stats = pd.concat([train_stats, val_stats, test_stats], ignore_index=True)

# Optional rounding for cleaner output
all_stats = all_stats.round(4)
all_stats.to_csv("./standardization_stats.csv", index=False)

#Printing out the lengths of each created set
print("Train:", len(features_train))
print("Validation:", len(features_val))
print("Test:", len(features_test))


In [ ]:
#Defining the sigmoid activation function, it is the same one that was used in the first lab work
def sigmoid(a):
    return 1 / (1 + np.exp(-a))

#Defining a function to calculate mean squared error, accuracy and predicted label for a given set of features, actual labels, weights and bias
def calculate_metrics(X, Y_actual, weights, bias):
    a = np.dot(X, weights) + bias # Calculating the linear combination of inputs and weights plus bias, this is the input to the sigmoid function, which will give us the predicted probabilities for each record in X. Shape of this structure is (n_samples,)
    Y_pred = sigmoid(a) # Applying the sigmoid function to all of the linear combinations, this gives us an array of predicted probabilities for each record in X. The shape is (n_samples,). Value explanations: the closer the value is to 0 or 1, the more confident the model is in its prediction, while values around 0.5 indicate uncertainty.
    
    mse = np.mean((Y_actual - Y_pred)**2) # Calculating the mean squared error (MSE) between the actual label values and the predicted probabilities. This way we can see how close the predictions are to the actual labels, the lower the MSE, the better the predictions are. The shape of this value is a single scalar.

    Y_round = np.round(Y_pred) #Here we round the  probabilities to align with a class (0 or 1), this way we can calculate the accuracy of the predictions, the shape of this structure is (n_samples,).
    accuracy = np.mean(Y_round == Y_actual) #We compare the arrays, index to index, to see how many predictions are correct (where the rounded predicted label matches the actual label), and then we take the mean of this boolean array to get the overall accuracy as a percentage. The shape of this value is a single scalar.

    return mse, accuracy, Y_round #Returning the calculated values of MSE, Accuracy and the rounded predicted labels

In [ ]:
def train_sigmoid_neuron(features_train, labels_train, features_val, labels_val,
                         learning_rate, epochs, mode='stochastic', seed = 20260326):

    np.random.seed(seed) #Setting a random seed for reproducibility in this case it is passed as the current date in YYMMDD format, but it can be any integer value
    
    n_features = features_train.shape[1] #Number of features in the training data, this is used to initialize the weights vector with the correct size
    weights = np.random.uniform(-0.5, 0.5, n_features) #Initializing the random weights from the (-0.5, 0.5) range, using uniform so the random values stay small and match the description in the report
    bias = np.random.uniform(-0.5, 0.5)

    labels_train = labels_train.flatten() #Uniform format - > making the 2D vector into 1D vector, this way we can access the labels with labels_train[i] instead of labels_train[i][0]
    labels_val = labels_val.flatten() #Uniform format - > making the 2D vector into 1D vector, this way we can access the labels with labels_val[i] instead of labels_val[i][0]

    history = { #Dictionary to store the history of training and validation MSE and accuracy for each epoch, this will be used for plotting the learning curves later
        'train_mse': [],
        'val_mse': [],
        'train_acc': [],
        'val_acc': []
    }

    m = len(features_train) #Number of training samples, this is used to calculate the average gradient in batch mode and to iterate through the training samples in both modes
    start_time = time.time() #Starting the timer to measure the training time, this will be used to compare the time taken for stochastic and batch gradient descent

    best_val_acc = -1 #Tracking the best validation accuracy seen during training
    best_val_mse = float("inf") #Tracking the best validation MSE seen during training
    best_epoch = 0 #Tracking the epoch where the best validation result was achieved
    best_weights = weights.copy() #Saving the best weights according to validation results
    best_bias = bias #Saving the best bias according to validation results

    for epoch in range(epochs): #Since we are not using any early stoppage criteria, we will be running the passed value of epochs, this can be tuned to find the optimal number of epochs, but it will not be implemented here.
 
        if mode == 'batch': #Batch gradient descent implementation
            grad_w = np.zeros(n_features) #Initializing the gradient for weights to zero, this will be used to accumulate the gradients for all training samples in the batch
            grad_b = 0 #Initializing the gradient for bias to zero, this will be used to accumulate the gradients for all training samples in the batch

            for i in range(m): #Iterating through each record in the batch and calculating the predicted output, error and gradients for weights and bias, and accumulating the gradients for the entire batch 
                y_pred = sigmoid(np.dot(features_train[i], weights) + bias)
                t = labels_train[i]

                delta = (y_pred - t) * y_pred * (1 - y_pred)

                grad_w += delta * features_train[i]
                grad_b += delta

            weights -= learning_rate * grad_w / m
            bias -= learning_rate * grad_b / m

        else:  # stochastic
            indices = np.random.permutation(m)

            for i in indices:
                y_pred = sigmoid(np.dot(features_train[i], weights) + bias)
                t = labels_train[i]

                delta = (y_pred - t) * y_pred * (1 - y_pred)

                weights -= learning_rate * delta * features_train[i]
                bias -= learning_rate * delta

        #Calculating the metrics after each epoch using the updated parameters
        train_mse, train_acc, _ = calculate_metrics(features_train, labels_train, weights, bias)
        val_mse, val_acc, _ = calculate_metrics(features_val, labels_val, weights, bias)

        history['train_mse'].append(train_mse)
        history['val_mse'].append(val_mse)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        #Saving the best model according to validation results. Primary criterion is validation accuracy, secondary criterion is validation MSE
        if (val_acc > best_val_acc) or (val_acc == best_val_acc and val_mse < best_val_mse):
            best_val_acc = val_acc
            best_val_mse = val_mse
            best_epoch = epoch + 1
            best_weights = weights.copy()
            best_bias = bias

    training_time = time.time() - start_time

    return best_weights, best_bias, history, training_time, best_epoch, best_val_mse, best_val_acc


In [ ]:
lr = 0.01
epochs = 500

# Stochastic
s_weights, s_bias, s_history, s_time, s_best_epoch, s_best_val_mse, s_best_val_acc = train_sigmoid_neuron(
    features_train, labels_train, features_val, labels_val,
    lr, epochs, mode='stochastic'
)

# Batch
b_weights, b_bias, b_history, b_time, b_best_epoch, b_best_val_mse, b_best_val_acc = train_sigmoid_neuron(
    features_train, labels_train, features_val, labels_val,
    lr, epochs, mode='batch'
)

print("Stochastic time:", s_time)
print("Batch time:", b_time)

print("Stochastic best epoch:", s_best_epoch)
print("Batch best epoch:", b_best_epoch)

print("Stochastic best validation MSE:", s_best_val_mse)
print("Batch best validation MSE:", b_best_val_mse)

print("Stochastic best validation accuracy:", s_best_val_acc)
print("Batch best validation accuracy:", b_best_val_acc)


In [ ]:
plt.plot(s_history['train_mse'], label='Train MSE Stochastic')
plt.plot(s_history['val_mse'], label='Val MSE Stochastic')
plt.plot(b_history['train_mse'], label='Train MSE Batch')
plt.plot(b_history['val_mse'], label='Val MSE Batch')

plt.xlabel("Epochs")
plt.ylabel("MSE")
plt.title("MSE vs Epochs")
plt.legend()
plt.show()

In [ ]:
plt.plot(s_history['train_acc'], label='Train Acc Stochastic')
plt.plot(s_history['val_acc'], label='Val Acc Stochastic')
plt.plot(b_history['train_acc'], label='Train Acc Batch')
plt.plot(b_history['val_acc'], label='Val Acc Batch')

plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.title("Accuracy vs Epochs")
plt.legend()
plt.show()

In [ ]:
learning_rates = [0.001, 0.01, 0.1]
results = []

best_model = None #This stores the best overall model, based on validation datasets results

for mode in ['stochastic', 'batch']:
    for lr in learning_rates:
        w, b, hist, t, best_epoch, best_val_mse, best_val_acc = train_sigmoid_neuron(
            features_train, labels_train, features_val, labels_val,
            lr, epochs, mode=mode
        )

        results.append([mode, lr, best_epoch, best_val_mse, best_val_acc, t, w.copy(), b])

        #Selecting the overall best model by validation results only. Test data is not used here
        if best_model is None:
            best_model = {
                "mode": mode,
                "lr": lr,
                "best_epoch": best_epoch,
                "best_val_mse": best_val_mse,
                "best_val_acc": best_val_acc,
                "weights": w.copy(),
                "bias": b
            }
        elif (best_val_acc > best_model["best_val_acc"]) or \
             (best_val_acc == best_model["best_val_acc"] and best_val_mse < best_model["best_val_mse"]):
            best_model = {
                "mode": mode,
                "lr": lr,
                "best_epoch": best_epoch,
                "best_val_mse": best_val_mse,
                "best_val_acc": best_val_acc,
                "weights": w.copy(),
                "bias": b
            }

results_df = pd.DataFrame(results, columns=["Mode", "LR", "Best Epoch", "Val MSE", "Val Acc", "Time", "Weights", "Bias"])
print("\nLearning Rate Results:")
print(results_df[["Mode", "LR", "Best Epoch", "Val MSE", "Val Acc", "Time"]])

In [ ]:
#Testing is performed once, after the best model has been chosen using validation data
test_mse_best, test_acc_best, test_pred_best = calculate_metrics(
    features_test, labels_test.flatten(), best_model["weights"], best_model["bias"]
)

print("\nBEST MODEL SELECTED USING VALIDATION DATA:")
print("Mode:", best_model["mode"])
print("Learning rate:", best_model["lr"])
print("Best epoch:", best_model["best_epoch"])
print("Validation MSE:", best_model["best_val_mse"])
print("Validation Accuracy:", best_model["best_val_acc"])

print("\nFINAL TEST RESULTS:")
print("MSE:", test_mse_best)
print("Accuracy:", test_acc_best)

print("\nBest weights:")
print(best_model["weights"])
print("Best bias:", best_model["bias"])

In [ ]:
print("\nTest predictions vs Actual:")
for i in range(len(features_test)):
    print("Pred:", int(test_pred_best[i]), "Actual:", int(labels_test[i]))